# 08 · Descomposición residencia–movilidad de la accesibilidad — Bogotá 2023

**Objetivo.** Separar la brecha de tiempo medio de primer paso (FPT) hacia empleo,
educación y salud en tres componentes: composición residencial, kernel de movilidad e
interacción. La unidad de remuestreo continúa siendo el hogar.

Para un grupo $g$, sea $M(P,\mu;T)$ el FPT medio hacia el conjunto objetivo $T$,
calculado con kernel $P$ y distribución residencial inicial $\mu$, condicionada a iniciar
fuera de $T$. Se evalúa el diseño factorial

$$
B=M(P_0,\mu_0),\quad C=M(P_0,\mu_g),\quad
K=M(P_g,\mu_0),\quad G=M(P_g,\mu_g).
$$

La identidad exacta es

$$
G-B=(C-B)+(K-B)+(G-C-K+B).
$$

También se reporta la asignación de Shapley para dos factores, que divide la interacción
por mitades y evita privilegiar un orden de sustitución. Los resultados son descriptivos,
no causales.


In [ ]:
%pip -q install pandas pyarrow numpy scipy matplotlib seaborn


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import os
import re
import shutil
import unicodedata
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path("/content/colombia-stochastic-access")
RAW = ROOT / "data/raw/bogota_em2023"
INTERIM = ROOT / "data/interim/bogota_em2023_gate9"
OUT = ROOT / "outputs/bogota_em2023/gate9"
DRIVE_BASE = Path("/content/drive/MyDrive/colombia-stochastic-access-data/bogota_em2023")
for directory in (RAW, INTERIM, OUT):
    directory.mkdir(parents=True, exist_ok=True)

SOURCE_URL = (
    "https://observatorio.movilidadbogota.gov.co/sites/default/files/"
    "2025-09/05_Base%20datos%20procesada%20EODH.zip"
)
PURPOSES = ("employment", "education", "health")
TARGET_DEFINITION = "core_50"
PRIMARY_TAU = 20.0
BOOTSTRAP_REPLICATES = 50
RANDOM_SEED = 202309

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc:
    print(f"Entorno fuera de Colab o Drive no montado: {exc}")

gate4_dir = Path(os.environ.get("GATE4_DIR_OVERRIDE", DRIVE_BASE / "results/gate4"))
gate6_dir = Path(os.environ.get("GATE6_DIR_OVERRIDE", DRIVE_BASE / "results/gate6"))
gate8_dir = Path(os.environ.get("GATE8_DIR_OVERRIDE", DRIVE_BASE / "results/gate8"))
EDGE_PATH = gate4_dir / "transition_edges_utam.parquet"
TARGET_PATH = gate6_dir / "purpose_target_sets.csv"
GATE8_PATH = gate8_dir / "gate_8.json"

missing = [str(path) for path in (EDGE_PATH, TARGET_PATH, GATE8_PATH) if not path.exists()]
if missing:
    raise FileNotFoundError("Faltan insumos validados: " + ", ".join(missing))

gate8 = json.loads(GATE8_PATH.read_text(encoding="utf-8"))
gate8_status = gate8.get("gate", gate8)
if not bool(gate8_status.get("gate_8_passed", False)):
    raise RuntimeError("La Compuerta 8 no fue aprobada.")

print("Kernel poblacional:", EDGE_PATH)
print("Destinos:", TARGET_PATH)
print("Salida:", OUT)


## 1. Recuperación de personas y viajes

La distribución residencial se estima desde el módulo de **personas**, no desde los
orígenes de los viajes. Así se evita que quienes realizan más viajes tengan mayor peso
residencial. Se usa `fexp_per_5anos` y `cod_utam_hg`; los kernels conservan `fexp_vj`.


In [ ]:
def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def valid_zip(path):
    path = Path(path)
    return path.exists() and path.stat().st_size > 0 and zipfile.is_zipfile(path)


def upload_archive(destination):
    from google.colab import files
    print("Descargue el ZIP oficial si no lo conserva:")
    print(SOURCE_URL)
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Suba exactamente un archivo ZIP.")
    name, content = next(iter(uploaded.items()))
    temporary = destination.with_suffix(".zip.upload")
    temporary.write_bytes(content)
    del content
    if not valid_zip(temporary):
        temporary.unlink(missing_ok=True)
        raise ValueError(f"{name} no es un ZIP válido.")
    temporary.replace(destination)
    return "manual_upload"


archive = RAW / "processed_edoh.zip"
drive_archive = DRIVE_BASE / "processed_edoh.zip"
if valid_zip(archive):
    retrieval_method = "runtime_cache"
elif valid_zip(drive_archive):
    shutil.copy2(drive_archive, archive)
    retrieval_method = "google_drive_cache"
else:
    retrieval_method = upload_archive(archive)
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    shutil.copy2(archive, drive_archive)


def safe_extract_recursive(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    queue = [(Path(zip_path), destination)]
    visited = set()
    while queue:
        current, target = queue.pop(0)
        signature = (str(current.resolve()), str(target.resolve()))
        if signature in visited:
            continue
        visited.add(signature)
        root = target.resolve()
        with zipfile.ZipFile(current) as zipped:
            for member in zipped.infolist():
                resolved = (target / member.filename).resolve()
                if root != resolved and root not in resolved.parents:
                    raise ValueError(f"Ruta insegura en ZIP: {member.filename}")
            zipped.extractall(target)
        for nested in target.rglob("*.zip"):
            nested_target = nested.parent / nested.stem
            if (str(nested.resolve()), str(nested_target.resolve())) not in visited:
                queue.append((nested, nested_target))


safe_extract_recursive(archive, INTERIM / "processed_edoh")
source_manifest = pd.DataFrame([{
    "source_id": "processed_edoh",
    "url": SOURCE_URL,
    "path": str(archive),
    "bytes": int(archive.stat().st_size),
    "sha256": sha256(archive),
    "retrieval_method": retrieval_method,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
}])
source_manifest.to_csv(OUT / "source_manifest.csv", index=False)
display(source_manifest)


In [ ]:
def normalize(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(char for char in value if not unicodedata.combining(char))
    return re.sub(r"[^a-z0-9]+", "_", value.lower().strip()).strip("_")


def clean_key(series):
    cleaned = series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
    invalid = cleaned.map(normalize).isin(
        ["", "nan", "none", "na", "no_aplica", "sin_informacion"]
    )
    return cleaned.mask(invalid)


def canonical_utam(series):
    cleaned = clean_key(series)
    def canonical(value):
        if pd.isna(value):
            return pd.NA
        text = str(value).strip().upper().replace(" ", "")
        upr = re.fullmatch(r"UPR0*(\d+)", text)
        if upr:
            return f"UPR{int(upr.group(1)):04d}"
        utam = re.fullmatch(r"(?:UTAM)?0*(\d+)", text)
        if utam:
            return f"UTAM{int(utam.group(1)):03d}"
        return text
    return cleaned.map(canonical).astype("string")


def parse_number(series):
    raw = series.astype("string").str.strip()
    if float(raw.str.contains(",", regex=False, na=False).mean()) > 0.05:
        raw = raw.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(raw, errors="coerce")


def csv_format(path):
    for encoding in ("utf-8-sig", "latin-1"):
        try:
            with path.open("r", encoding=encoding, errors="strict") as handle:
                sample = handle.read(32768)
            return encoding, csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
        except (UnicodeDecodeError, csv.Error):
            continue
    raise ValueError(f"No fue posible detectar el formato: {path}")


def find_module(token, excluded=()):
    paths = list((INTERIM / "processed_edoh").rglob("*.csv"))
    candidates = [p for p in paths if token in normalize(p.stem)
                  and not any(x in normalize(p.stem) for x in excluded)]
    if not candidates:
        raise FileNotFoundError(f"No se encontró el módulo CSV: {token}")
    return sorted(candidates, key=lambda p: (len(p.parts), len(p.name)))[0]


def read_selected(path, required):
    encoding, delimiter = csv_format(path)
    header = pd.read_csv(path, sep=delimiter, encoding=encoding, nrows=0)
    mapping = {normalize(column): column for column in header.columns}
    absent = sorted(set(required) - set(mapping))
    if absent:
        raise KeyError(f"Faltan columnas en {path.name}: {absent}")
    frame = pd.read_csv(
        path, sep=delimiter, encoding=encoding,
        usecols=[mapping[column] for column in required], dtype="string", low_memory=False,
    )
    frame.columns = [normalize(column) for column in frame.columns]
    return frame


trip_path = find_module("viaje", excluded=("etapa",))
person_path = find_module("persona")
trips = read_selected(
    trip_path, ["key_hg", "utam_ori", "utam_des", "fexp_vj", "sexo", "estra_hg"]
)
persons = read_selected(
    person_path, ["key_hg", "cod_utam_hg", "fexp_per_5anos", "sexo", "estra_hg"]
)

trips["household"] = clean_key(trips["key_hg"])
trips["utam_ori"] = canonical_utam(trips["utam_ori"])
trips["utam_des"] = canonical_utam(trips["utam_des"])
trips["weight"] = parse_number(trips["fexp_vj"])
trips["sex_group"] = trips["sexo"].map(normalize)
trips["stratum_group"] = clean_key(trips["estra_hg"])
trips = trips[
    trips[["household", "utam_ori", "utam_des"]].notna().all(axis=1)
    & trips["weight"].gt(0)
].reset_index(drop=True)

persons["household"] = clean_key(persons["key_hg"])
persons["residence_utam"] = canonical_utam(persons["cod_utam_hg"])
persons["weight"] = parse_number(persons["fexp_per_5anos"])
persons["sex_group"] = persons["sexo"].map(normalize)
persons["stratum_group"] = clean_key(persons["estra_hg"])
persons = persons[
    persons[["household", "residence_utam"]].notna().all(axis=1)
    & persons["weight"].gt(0)
].reset_index(drop=True)

print("Viajes válidos:", f"{len(trips):,}")
print("Personas válidas:", f"{len(persons):,}")
print("Archivo de personas:", person_path.name)


## 2. Soporte común, kernels y distribuciones residenciales

Los nodos se fijan con la Gate 4 y los objetivos con la Gate 6. Una residencia fuera del
soporte del kernel no se reasigna: se cuantifica y excluye. Cada distribución residencial
se normaliza nuevamente para cada objetivo después de condicionar a iniciar fuera de él.


In [ ]:
edges = pd.read_parquet(EDGE_PATH).copy()
for column in ("utam_ori", "utam_des"):
    edges[column] = canonical_utam(edges[column])
edges = edges.dropna(subset=["utam_ori", "utam_des"])
nodes = sorted(set(edges["utam_ori"]) | set(edges["utam_des"]))
node_to_index = {node: index for index, node in enumerate(nodes)}
n_nodes = len(nodes)

P0 = np.zeros((n_nodes, n_nodes), dtype=float)
for row in edges.itertuples(index=False):
    P0[node_to_index[row.utam_ori], node_to_index[row.utam_des]] = float(
        row.transition_probability
    )
if not np.allclose(P0.sum(axis=1), 1.0, atol=1e-10):
    raise RuntimeError("El kernel poblacional no es estocástico.")

trip_in_support = trips["utam_ori"].isin(node_to_index) & trips["utam_des"].isin(node_to_index)
person_in_support = persons["residence_utam"].isin(node_to_index)
support_audit = pd.DataFrame([{
    "trip_row_support_share": float(trip_in_support.mean()),
    "trip_weight_support_share": float(trips.loc[trip_in_support, "weight"].sum() / trips["weight"].sum()),
    "person_row_residential_support_share": float(person_in_support.mean()),
    "person_weight_residential_support_share": float(
        persons.loc[person_in_support, "weight"].sum() / persons["weight"].sum()
    ),
}])
support_audit.to_csv(OUT / "residential_support_audit.csv", index=False)
display(support_audit)

trips = trips.loc[trip_in_support].reset_index(drop=True)
persons = persons.loc[person_in_support].reset_index(drop=True)
origin_index = trips["utam_ori"].map(node_to_index).to_numpy(dtype=int)
destination_index = trips["utam_des"].map(node_to_index).to_numpy(dtype=int)
base_trip_weight = trips["weight"].to_numpy(dtype=float)
residence_index = persons["residence_utam"].map(node_to_index).to_numpy(dtype=int)
base_person_weight = persons["weight"].to_numpy(dtype=float)

target_table = pd.read_csv(TARGET_PATH, dtype={"utam": "string"})
target_table["utam"] = target_table["utam"].astype("string").str.strip().str.upper()
target_sets = {}
for purpose in PURPOSES:
    codes = set(target_table.loc[
        target_table["purpose_group"].eq(purpose)
        & target_table["target_definition"].eq(TARGET_DEFINITION), "utam"
    ])
    missing_codes = sorted(codes - set(nodes))
    if missing_codes:
        raise RuntimeError(f"Destinos fuera del kernel para {purpose}: {missing_codes}")
    target_sets[purpose] = np.array(sorted(node_to_index[code] for code in codes), dtype=int)

expected_groups = {
    "sex:mujer": ("sex", "mujer"),
    "sex:hombre": ("sex", "hombre"),
}
for level in map(str, range(1, 7)):
    expected_groups[f"stratum:{level}"] = ("stratum", level)


def group_mask(frame, dimension, level):
    column = "sex_group" if dimension == "sex" else "stratum_group"
    return frame[column].eq(level).fillna(False).to_numpy(dtype=bool)


trip_masks = {gid: group_mask(trips, dim, lev) for gid, (dim, lev) in expected_groups.items()}
person_masks = {gid: group_mask(persons, dim, lev) for gid, (dim, lev) in expected_groups.items()}
group_inventory = pd.DataFrame([{
    "group_id": gid,
    "dimension": dim,
    "level": lev,
    "sampled_trips": int(trip_masks[gid].sum()),
    "sampled_persons": int(person_masks[gid].sum()),
    "weighted_persons": float(persons.loc[person_masks[gid], "weight"].sum()),
    "trip_households": int(trips.loc[trip_masks[gid], "household"].nunique()),
    "person_households": int(persons.loc[person_masks[gid], "household"].nunique()),
} for gid, (dim, lev) in expected_groups.items()])
group_inventory.to_csv(OUT / "decomposition_group_inventory.csv", index=False)
display(group_inventory)


## 3. Descomposición factorial puntual

Los efectos principales usan como referencia $(P_0,\mu_0)$. La interacción registra la
parte no aditiva. Para facilitar comparaciones, `shapley_residential` y `shapley_kernel`
reparten esa interacción por mitades y suman exactamente la brecha total.


In [ ]:
def regularized_kernel(mask, tau, row_multiplier=None):
    selected = np.flatnonzero(mask)
    original = base_trip_weight[selected]
    multiplicity = (
        np.ones(len(selected), dtype=float)
        if row_multiplier is None else row_multiplier[selected].astype(float)
    )
    weights = original * multiplicity
    squared = original**2 * multiplicity
    origins = origin_index[selected]
    destinations = destination_index[selected]
    positive = weights > 0
    weights, squared = weights[positive], squared[positive]
    origins, destinations = origins[positive], destinations[positive]

    W = np.zeros((n_nodes, n_nodes), dtype=float)
    np.add.at(W, (origins, destinations), weights)
    out = W.sum(axis=1)
    sum_sq = np.bincount(origins, weights=squared, minlength=n_nodes)
    ess = np.divide(out**2, sum_sq, out=np.zeros(n_nodes), where=sum_sq > 0)
    empirical = np.zeros_like(W)
    observed = out > 0
    empirical[observed] = W[observed] / out[observed, None]
    omega = ess / (ess + float(tau))
    return omega[:, None] * empirical + (1.0 - omega[:, None]) * P0


def residential_distribution(mask=None, row_multiplier=None):
    if mask is None:
        mask = np.ones(len(persons), dtype=bool)
    selected = np.flatnonzero(mask)
    multiplicity = (
        np.ones(len(selected), dtype=float)
        if row_multiplier is None else row_multiplier[selected].astype(float)
    )
    weights = base_person_weight[selected] * multiplicity
    mu = np.bincount(residence_index[selected], weights=weights, minlength=n_nodes).astype(float)
    if not np.isfinite(mu).all() or mu.sum() <= 0:
        raise RuntimeError("Distribución residencial vacía o no finita.")
    return mu / mu.sum()


def first_passage_by_origin(P, targets):
    target_mask = np.zeros(n_nodes, dtype=bool)
    target_mask[targets] = True
    transient = np.flatnonzero(~target_mask)
    Q = P[np.ix_(transient, transient)]
    mean = np.linalg.solve(np.eye(len(transient)) - Q, np.ones(len(transient)))
    full = np.zeros(n_nodes, dtype=float)
    full[transient] = mean
    return full


def conditional_mean(h, mu, targets):
    eligible = np.ones(n_nodes, dtype=bool)
    eligible[targets] = False
    mass = float(mu[eligible].sum())
    if mass <= 0:
        raise RuntimeError("No hay masa residencial fuera del conjunto objetivo.")
    return float((mu[eligible] / mass) @ h[eligible])


def factorial_values(h0, hg, mu0, mug, targets):
    B = conditional_mean(h0, mu0, targets)
    C = conditional_mean(h0, mug, targets)
    K = conditional_mean(hg, mu0, targets)
    G = conditional_mean(hg, mug, targets)
    composition = C - B
    kernel = K - B
    interaction = G - C - K + B
    total = G - B
    return {
        "population_baseline": B,
        "residence_only": C,
        "kernel_only": K,
        "group_observed": G,
        "residential_main": composition,
        "kernel_main": kernel,
        "interaction": interaction,
        "shapley_residential": composition + 0.5 * interaction,
        "shapley_kernel": kernel + 0.5 * interaction,
        "total_gap": total,
        "closure_error": total - composition - kernel - interaction,
    }


mu0 = residential_distribution()
group_mu = {gid: residential_distribution(person_masks[gid]) for gid in expected_groups}
group_kernels = {gid: regularized_kernel(trip_masks[gid], PRIMARY_TAU) for gid in expected_groups}

point_rows = []
origin_rows = []
h0_by_purpose = {}
hg_by_group_purpose = {}
for purpose in PURPOSES:
    targets = target_sets[purpose]
    h0 = first_passage_by_origin(P0, targets)
    h0_by_purpose[purpose] = h0
    for gid, (dimension, level) in expected_groups.items():
        hg = first_passage_by_origin(group_kernels[gid], targets)
        hg_by_group_purpose[(gid, purpose)] = hg
        values = factorial_values(h0, hg, mu0, group_mu[gid], targets)
        point_rows.append({
            "group_id": gid, "dimension": dimension, "level": level,
            "purpose_group": purpose, "target_definition": TARGET_DEFINITION,
            **values,
        })
        for index, node in enumerate(nodes):
            origin_rows.append({
                "group_id": gid, "purpose_group": purpose, "utam": node,
                "population_fpt_steps": float(h0[index]),
                "group_fpt_steps": float(hg[index]),
                "population_residential_probability": float(mu0[index]),
                "group_residential_probability": float(group_mu[gid][index]),
            })

point = pd.DataFrame(point_rows)
by_origin = pd.DataFrame(origin_rows)
point.to_csv(OUT / "residence_kernel_decomposition.csv", index=False)
by_origin.to_parquet(OUT / "decomposition_by_origin.parquet", index=False)
display(point)


## 4. Brecha mujer–hombre: descomposición simétrica

La comparación directa no usa a la población como categoría intermedia. La contribución
residencial promedia el cambio de residencia bajo ambos kernels; la contribución del
kernel promedia el cambio de kernel bajo ambas distribuciones residenciales.


In [ ]:
def sex_gap_values(h_w, h_m, mu_w, mu_m, targets):
    mm = conditional_mean(h_m, mu_m, targets)
    mw = conditional_mean(h_m, mu_w, targets)
    wm = conditional_mean(h_w, mu_m, targets)
    ww = conditional_mean(h_w, mu_w, targets)
    residential = 0.5 * ((mw - mm) + (ww - wm))
    kernel = 0.5 * ((wm - mm) + (ww - mw))
    total = ww - mm
    return {
        "male_total_fpt": mm,
        "female_total_fpt": ww,
        "female_minus_male_total": total,
        "residential_contribution": residential,
        "kernel_contribution": kernel,
        "closure_error": total - residential - kernel,
    }


sex_rows = []
for purpose in PURPOSES:
    values = sex_gap_values(
        hg_by_group_purpose[("sex:mujer", purpose)],
        hg_by_group_purpose[("sex:hombre", purpose)],
        group_mu["sex:mujer"], group_mu["sex:hombre"], target_sets[purpose],
    )
    sex_rows.append({"purpose_group": purpose, **values})

sex_decomposition = pd.DataFrame(sex_rows)
sex_decomposition.to_csv(OUT / "female_male_decomposition.csv", index=False)
display(sex_decomposition)


## 5. Bootstrap Poisson conjunto por hogar

Una misma multiplicidad Poisson(1) se aplica a todas las personas y viajes del hogar.
Esto conserva la dependencia entre composición residencial y movilidad. Cincuenta
réplicas constituyen una compuerta computacional; para el manuscrito se usarán al menos 500.


In [ ]:
all_households = pd.Index(sorted(
    set(trips["household"].dropna().astype(str))
    | set(persons["household"].dropna().astype(str))
))
household_to_code = {household: index for index, household in enumerate(all_households)}
trip_household_codes = trips["household"].astype(str).map(household_to_code).to_numpy(dtype=int)
person_household_codes = persons["household"].astype(str).map(household_to_code).to_numpy(dtype=int)

rng = np.random.default_rng(RANDOM_SEED)
bootstrap_rows = []
sex_bootstrap_rows = []
for replicate in range(1, BOOTSTRAP_REPLICATES + 1):
    household_multiplier = rng.poisson(1, len(all_households))
    trip_multiplier = household_multiplier[trip_household_codes]
    person_multiplier = household_multiplier[person_household_codes]
    mu0_boot = residential_distribution(row_multiplier=person_multiplier)
    group_mu_boot = {
        gid: residential_distribution(person_masks[gid], person_multiplier)
        for gid in expected_groups
    }
    group_kernel_boot = {
        gid: regularized_kernel(trip_masks[gid], PRIMARY_TAU, trip_multiplier)
        for gid in expected_groups
    }

    for purpose in PURPOSES:
        targets = target_sets[purpose]
        h0 = h0_by_purpose[purpose]  # P0 se mantiene fijo, como en Gate 8.
        h_boot = {
            gid: first_passage_by_origin(P_boot, targets)
            for gid, P_boot in group_kernel_boot.items()
        }
        for gid, (dimension, level) in expected_groups.items():
            values = factorial_values(h0, h_boot[gid], mu0_boot, group_mu_boot[gid], targets)
            bootstrap_rows.append({
                "replicate": int(replicate), "group_id": gid,
                "dimension": dimension, "level": level,
                "purpose_group": purpose, **values,
            })
        sex_values = sex_gap_values(
            h_boot["sex:mujer"], h_boot["sex:hombre"],
            group_mu_boot["sex:mujer"], group_mu_boot["sex:hombre"], targets,
        )
        sex_bootstrap_rows.append({
            "replicate": int(replicate), "purpose_group": purpose, **sex_values,
        })
    if replicate % 10 == 0:
        print(f"Bootstrap {replicate}/{BOOTSTRAP_REPLICATES}")

bootstrap = pd.DataFrame(bootstrap_rows)
sex_bootstrap = pd.DataFrame(sex_bootstrap_rows)
bootstrap.to_parquet(OUT / "decomposition_household_bootstrap.parquet", index=False)
sex_bootstrap.to_parquet(OUT / "female_male_decomposition_bootstrap.parquet", index=False)

effect_columns = [
    "residential_main", "kernel_main", "interaction",
    "shapley_residential", "shapley_kernel", "total_gap",
]
interval_rows = []
for keys, frame in bootstrap.groupby(
    ["group_id", "dimension", "level", "purpose_group"], observed=True
):
    for effect in effect_columns:
        values = frame[effect]
        interval_rows.append({
            "group_id": keys[0], "dimension": keys[1], "level": keys[2],
            "purpose_group": keys[3], "effect": effect,
            "bootstrap_mean": float(values.mean()),
            "bootstrap_sd": float(values.std()),
            "ci_lower": float(values.quantile(0.025)),
            "ci_upper": float(values.quantile(0.975)),
        })
intervals = pd.DataFrame(interval_rows)
intervals.to_csv(OUT / "decomposition_bootstrap_intervals.csv", index=False)

sex_effects = [
    "female_minus_male_total", "residential_contribution", "kernel_contribution"
]
sex_interval_rows = []
for purpose, frame in sex_bootstrap.groupby("purpose_group", observed=True):
    point_row = sex_decomposition.set_index("purpose_group").loc[purpose]
    for effect in sex_effects:
        values = frame[effect]
        sex_interval_rows.append({
            "purpose_group": purpose, "effect": effect,
            "point_estimate": float(point_row[effect]),
            "bootstrap_mean": float(values.mean()),
            "bootstrap_sd": float(values.std()),
            "ci_lower": float(values.quantile(0.025)),
            "ci_upper": float(values.quantile(0.975)),
        })
sex_intervals = pd.DataFrame(sex_interval_rows)
sex_intervals.to_csv(OUT / "female_male_decomposition_intervals.csv", index=False)
display(sex_intervals)


## 6. Visualización

El primer panel presenta la brecha total grupo–población. El segundo separa la brecha
mujer–hombre mediante Shapley. El tercero muestra, por estrato, las contribuciones
residencial y del kernel; ambas suman la brecha total de cada estrato.


In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
palette = dict(zip(PURPOSES, sns.color_palette("colorblind", len(PURPOSES))))
fig, axes = plt.subplots(1, 3, figsize=(18, 5.4))

sex_point = point[point["dimension"].eq("sex")].copy()
sns.barplot(
    data=sex_point, x="level", y="total_gap", hue="purpose_group",
    palette=palette, ax=axes[0], order=["hombre", "mujer"],
)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set(title="Brecha frente a población", xlabel="Sexo", ylabel="Diferencia de FPT")

sex_long = sex_decomposition.melt(
    id_vars="purpose_group",
    value_vars=["residential_contribution", "kernel_contribution"],
    var_name="component", value_name="contribution",
)
sns.barplot(
    data=sex_long, x="purpose_group", y="contribution", hue="component",
    palette=["#4C78A8", "#F58518"], ax=axes[1],
)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(title="Mujer − hombre", xlabel="Propósito", ylabel="Contribución al FPT")

stratum_point = point[point["dimension"].eq("stratum")].copy()
stratum_point["level_numeric"] = pd.to_numeric(stratum_point["level"])
for purpose in PURPOSES:
    subset = stratum_point[stratum_point["purpose_group"].eq(purpose)].sort_values("level_numeric")
    axes[2].plot(
        subset["level_numeric"], subset["shapley_residential"],
        marker="o", color=palette[purpose], label=f"{purpose}: residencia",
    )
    axes[2].plot(
        subset["level_numeric"], subset["shapley_kernel"],
        marker="s", linestyle="--", color=palette[purpose], label=f"{purpose}: kernel",
    )
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set_xticks(range(1, 7))
axes[2].set(title="Contribuciones por estrato", xlabel="Estrato", ylabel="Contribución al FPT")
axes[2].legend(fontsize=8, ncol=2)

fig.tight_layout()
fig.savefig(OUT / "residence_kernel_decomposition.png", dpi=180, bbox_inches="tight")
plt.show()


## 7. Compuerta 9

La compuerta verifica soporte, cierre algebraico, finitud y completitud computacional.
No exige significancia ni una dirección específica de los efectos.


In [ ]:
thresholds = {
    "minimum_weighted_residential_support_share": 0.90,
    "maximum_decomposition_closure_error": 1e-10,
    "minimum_sampled_persons_per_group": 300,
}
checks = {
    "gate8_dependency_valid": bool(gate8_status.get("gate_8_passed", False)),
    "expected_groups_present": bool(len(group_inventory) == 8),
    "person_sample_size_valid": bool(
        group_inventory["sampled_persons"].ge(
            thresholds["minimum_sampled_persons_per_group"]
        ).all()
    ),
    "residential_support_valid": bool(
        support_audit.loc[0, "person_weight_residential_support_share"]
        >= thresholds["minimum_weighted_residential_support_share"]
    ),
    "residential_distributions_stochastic": bool(
        abs(mu0.sum() - 1.0) <= 1e-12
        and all(abs(mu.sum() - 1.0) <= 1e-12 for mu in group_mu.values())
    ),
    "point_estimates_finite": bool(
        np.isfinite(point[effect_columns + ["interaction"]].to_numpy()).all()
    ),
    "factorial_identity_valid": bool(
        point["closure_error"].abs().max()
        <= thresholds["maximum_decomposition_closure_error"]
    ),
    "sex_identity_valid": bool(
        sex_decomposition["closure_error"].abs().max()
        <= thresholds["maximum_decomposition_closure_error"]
    ),
    "bootstrap_complete": bool(
        bootstrap["replicate"].nunique() == BOOTSTRAP_REPLICATES
        and len(bootstrap) == BOOTSTRAP_REPLICATES * len(expected_groups) * len(PURPOSES)
        and len(sex_bootstrap) == BOOTSTRAP_REPLICATES * len(PURPOSES)
    ),
    "bootstrap_intervals_finite": bool(
        np.isfinite(intervals[["ci_lower", "ci_upper"]].to_numpy()).all()
        and np.isfinite(sex_intervals[["ci_lower", "ci_upper"]].to_numpy()).all()
    ),
}
checks["gate_9_passed"] = bool(all(checks.values()))

report = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "source_sha256": str(source_manifest.loc[0, "sha256"]),
    "estimand": "residential-kernel factorial decomposition of conditional mean FPT",
    "target_definition": TARGET_DEFINITION,
    "regularization_tau": float(PRIMARY_TAU),
    "bootstrap": {
        "method": "joint Poisson(1) household cluster bootstrap over persons and trips",
        "replicates": int(BOOTSTRAP_REPLICATES),
        "random_seed": int(RANDOM_SEED),
        "population_kernel": "fixed Gate 4 kernel",
    },
    "thresholds": {key: float(value) for key, value in thresholds.items()},
    "support_audit": json.loads(support_audit.to_json(orient="records")),
    "sex_decomposition": json.loads(sex_decomposition.to_json(orient="records")),
    "gate": checks,
}
(OUT / "gate_9.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8"
)
pd.Series(checks, name="result").to_csv(OUT / "gate_9_checks.csv")
display(pd.Series(checks, name="result"))

persistent_out = DRIVE_BASE / "results/gate9"
persistent_out.mkdir(parents=True, exist_ok=True)
for artifact in OUT.iterdir():
    if artifact.is_file():
        shutil.copy2(artifact, persistent_out / artifact.name)

print("Compuerta 9:", "APROBADA" if checks["gate_9_passed"] else "NO APROBADA")
print("Copia persistente:", persistent_out)


## Interpretación prevista

- Una contribución residencial positiva indica que la localización de las viviendas eleva
  el FPT respecto de la referencia; una negativa indica lo contrario.
- Una contribución positiva del kernel indica que los patrones de transición elevan el FPT
  manteniendo comparable la distribución inicial.
- La interacción cuantifica no aditividad, no un tercer mecanismo causal.
- La descomposición mujer–hombre es el contraste principal. Los estratos se presentan como
  gradiente exploratorio, especialmente porque los estratos 5–6 requieren mayor contracción.
- Si la Gate 9 pasa, se aumentará el bootstrap a 500–1000 réplicas y se prepararán las
  figuras y tablas del manuscrito.
